In [ ]:
import logging

from dotenv import load_dotenv
from langchain_core.documents import Document

from rag.ingestion.llama_parse_processor import process_document
from rag.ingestion.document_fetcher import DocumentFetcher
from rag.ingestion.document_tracker import DocumentTracker
from rag.ingestion.vector_store import QdrantManager
from utils.data_helpers import (
    initialize_metadata_data,
    initialize_stock_data,
)

load_dotenv()

logger = logging.getLogger(__name__)


await initialize_stock_data()
await initialize_metadata_data()

2025-11-19 20:11:55,725 - INFO - Stock data already initialized, skipping
2025-11-19 20:11:55,725 - INFO - Metadata already initialized, skipping


### Get Stocks from Nifty 50 Index

In [15]:
from rag.schemas.rag_schemas import DocumentMetadata
from utils.data_helpers import get_metadata_item_by_attachment_name

docs_to_test_sources = [
    "9b87cafe-1db5-4e8b-ac49-6ebcd54a8394.pdf",
    "88feb214-165a-4db1-b916-df4fbeae221c.pdf",
]
docs_to_test: list[DocumentMetadata] = []

for source in docs_to_test_sources:
    item = get_metadata_item_by_attachment_name(source)
    _doc = DocumentMetadata(
        filename=item.get("attachmentname", ""),
        fincode=item.get("fincode", 0),
        category=item.get("subcatname", ""),
        document_date=item.get("newsDt", "")[:10],  # Extract YYYY-MM-DD
        is_embedded=False,  # Will be updated by vector store check
    )
    docs_to_test.append(_doc)
docs_to_test

[DocumentMetadata(filename='9b87cafe-1db5-4e8b-ac49-6ebcd54a8394.pdf', fincode=103806, company_name=None, category='concall', document_date='2025-05-13', is_embedded=False, page_count=None),
 DocumentMetadata(filename='88feb214-165a-4db1-b916-df4fbeae221c.pdf', fincode=103806, company_name=None, category='concall', document_date='2025-11-04', is_embedded=False, page_count=None)]

In [16]:
import os

from joblib import Parallel, delayed

fetcher = DocumentFetcher()
doc_tracker = DocumentTracker()
qdrant_manager = QdrantManager() 

# Calculate optimal number of workers (75% of CPU cores)
max_workers = max(1, int(os.cpu_count() * 0.75))
logger.info(f"Using {max_workers} parallel workers (75% of {os.cpu_count()} cores)")

2025-11-19 20:11:55,746 - INFO - Initialized DocumentTracker with database at d:\work\finSharpe\clients\define-edge\repos\embed_docs\rag\data\embedded_docs.db
2025-11-19 20:11:56,046 - INFO - Initialized DocumentTracker with database at d:\work\finSharpe\clients\define-edge\repos\embed_docs\rag\data\embedded_docs.db
2025-11-19 20:11:56,046 - INFO - Initialized QdrantManager with collection 'company_files'
2025-11-19 20:11:56,046 - INFO - Using 15 parallel workers (75% of 20 cores)


In [ ]:
import asyncio
from typing import Any


# Wrapper function to process a single PDF using joblib
def process_pdf_wrapper(pdf_stream: Any, filename: str) -> tuple[list[Document], dict]:
    """process_document synchronously to use with joblib.
    
    Returns:
        Tuple of (documents, diagnostics)
    """
    diagnostics = {
        "filename": filename,
        "metadata_initialized": False,
        "stock_data_initialized": False,
        "error": None,
        "sample_metadata": None
    }
    
    try:
        # CRITICAL: Initialize metadata and stock data in THIS worker process
        # The main process's data is NOT shared with worker processes
        import asyncio
        from utils.data_helpers import (
            initialize_metadata_data,
            initialize_stock_data,
            is_metadata_initialized,
            is_stock_data_initialized,
        )
        
        # Create event loop for async initialization
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        
        try:
            # Initialize data in this worker process
            loop.run_until_complete(initialize_stock_data())
            loop.run_until_complete(initialize_metadata_data())
            
            diagnostics["metadata_initialized"] = is_metadata_initialized()
            diagnostics["stock_data_initialized"] = is_stock_data_initialized()
            
            # Process the PDF
            result = loop.run_until_complete(process_document(pdf_stream, file_type="pdf"))
            
            # Capture sample metadata from first document
            if result and len(result) > 0:
                diagnostics["sample_metadata"] = result[0].metadata
            
            return result, diagnostics
        finally:
            loop.close()
            
    except Exception as e:
        diagnostics["error"] = str(e)
        import traceback
        diagnostics["traceback"] = traceback.format_exc()
        return [], diagnostics


async def process_test_docs():
    """Process a single fincode: fetch, check existence, process, and embed documents."""
    # Set documents to test
    docs = docs_to_test

    try:
        # check which documents already exist in vector store
        exists = await doc_tracker.check_documents_exist([doc.filename for doc in docs])
        logger.info(f"Document existence check complete. Results: {exists}")
    except Exception as e:
        logger.error(f"Error checking document existence: {e}")
        return

    # Collect PDFs that need processing
    try:
        pdfs_to_process = []

        for doc in docs:
            if not exists[doc.filename]:
                try:
                    logger.info(
                        f"Document {doc.filename} does not exist in vector store. Proceeding to fetch."
                    )
                    pdf = await fetcher.get_pdf_doc_stream(doc.filename, doc.category)
                    file_size = len(pdf.stream.getbuffer()) / 1024 / 1024
                    logger.info(f"Document size: {file_size:.2f} MB - {doc.filename}")

                    pdfs_to_process.append((pdf, doc.filename))
                except Exception as e:
                    logger.error(f"Error fetching document {doc.filename}: {e}")
                    continue
    except Exception as e:
        logger.error(f"Error collecting PDFs: {e}")
        return

    # Process all PDFs in parallel using joblib (CPU-intensive operation)
    try:
        processed_docs: list[Document] = []

        if pdfs_to_process:
            logger.info(f"Processing {len(pdfs_to_process)} PDFs in parallel")

            # Use joblib to process PDFs in parallel
            results = Parallel(n_jobs=max_workers, backend="loky", verbose=10)(
                delayed(process_pdf_wrapper)(pdf, filename)
                for pdf, filename in pdfs_to_process
            )

            # Process results and log diagnostics
            for docs_result, diag in results:
                logger.info(f"\n--- Diagnostics for {diag['filename']} ---")
                logger.info(f"Metadata initialized: {diag['metadata_initialized']}")
                logger.info(f"Stock data initialized: {diag['stock_data_initialized']}")
                if diag['error']:
                    logger.error(f"Error: {diag['error']}")
                    logger.error(f"Traceback: {diag.get('traceback', 'N/A')}")
                if diag['sample_metadata']:
                    logger.info(f"Sample metadata: {diag['sample_metadata']}")
                logger.info("---\n")
                
                if docs_result:
                    processed_docs.extend(docs_result)

            logger.info(
                f"Processed {len(processed_docs)} document chunks from {len(pdfs_to_process)} PDFs"
            )
    except Exception as e:
        logger.error(f"Error processing PDFs: {e}")
        import traceback
        logger.error(traceback.format_exc())
        return

    # Select documents that need to be embedded
    try:
        docs_to_embed: list[Document] = []
        for doc in processed_docs:
            source = doc.metadata.get("source", "")
            if not exists.get(source, False):
                docs_to_embed.append(doc)

        logger.info(f"Found {len(docs_to_embed)} document chunks to embed")
    except Exception as e:
        logger.error(f"Error selecting documents to embed: {e}")
        return

    # Log the documents to embed
    for doc in docs_to_embed:
        metadata = doc.metadata
        logger.info(f"Document to embed: {doc.metadata.get('source', 'unknown source')}")
        logger.info(f"{doc.page_content[:100]}...")
        logger.info(f"category: {metadata.get('category', 'unknown')}")
        logger.info(f"document_date: {metadata.get('document_date', 'unknown')}")
        logger.info(f"fincode: {metadata.get('fincode', 'unknown')}")
        logger.info(f"ticker: {metadata.get('ticker', 'unknown')}")
        logger.info("---\n")
        


    # Embed documents
    # try:
    #     if isinstance(docs_to_embed, list) and len(docs_to_embed) > 0:
    #         result = await qdrant_manager.embed_documents(docs_to_embed)
    #         logger.info(
    #             f"Fincode {fincode} - Embedded {len(docs_to_embed)} chunks - Cost: ${result['estimated_cost_usd']:.4f}"
    #         )
    #     else:
    #         logger.info(f"No new documents to embed for fincode {fincode}. Skipping.")
    # except Exception as e:
    #     logger.error(f"Error embedding documents for fincode {fincode}: {e}")

In [18]:
# Process fincodes sequentially, but PDFs in parallel within each fincode
# This approach processes one fincode at a time, but uses all CPU cores for PDF processing
await process_test_docs()


2025-11-19 20:11:56,073 - INFO - Document existence check complete. Results: {'9b87cafe-1db5-4e8b-ac49-6ebcd54a8394.pdf': False, '88feb214-165a-4db1-b916-df4fbeae221c.pdf': False}
2025-11-19 20:11:56,079 - INFO - Document 9b87cafe-1db5-4e8b-ac49-6ebcd54a8394.pdf does not exist in vector store. Proceeding to fetch.


2025-11-19 20:11:56,907 - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=concall&attachmentName=9b87cafe-1db5-4e8b-ac49-6ebcd54a8394.pdf "HTTP/1.1 200 OK"
2025-11-19 20:11:56,993 - INFO - Document size: 0.22 MB - 9b87cafe-1db5-4e8b-ac49-6ebcd54a8394.pdf
2025-11-19 20:11:56,993 - INFO - Document 88feb214-165a-4db1-b916-df4fbeae221c.pdf does not exist in vector store. Proceeding to fetch.
2025-11-19 20:11:57,385 - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=concall&attachmentName=88feb214-165a-4db1-b916-df4fbeae221c.pdf "HTTP/1.1 200 OK"
2025-11-19 20:11:57,487 - INFO - Document size: 0.31 MB - 88feb214-165a-4db1-b916-df4fbeae221c.pdf
2025-11-19 20:11:57,487 - INFO - Processing 2 PDFs in parallel
[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.
[Parallel(n_jobs=15)]: Done   2 out of   2 | elapsed:   10.7s fini